# M9.8 · Morfolojik görüntü işleme

Kümeler kuramının diliyle görüntü işleme: yansıma ve öteleme, erozyon ve genişleme (dilation), ikilikleri, açma ve kapama, hit-or-miss dönüşümü, gri seviye (düz ve düz olmayan) morfoloji, morfolojik gradyan, top-hat ve bottom-hat. Saf Python'da ders kitabı tanımlarıyla ikili işlemleri yazıp OpenCV'nin genişlemeyi neden yansıtmadan hesapladığını yakalıyorsun; NumPy'da gri seviye işlemlerin hepsini `cv2.morphologyEx` ile birebir eşliyorsun; PyTorch'ta genişlemeyi `max` + toplama olarak kurup yapı elemanını gradyanla öğreniyorsun; en sonda van Herk / Gil-Werman algoritmasıyla pencere boyutundan bağımsız sürede çalışan dikdörtgen morfolojiyi vektörel yazıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m9-8-morphology.html](https://m-gonul.github.io/pytorch-lab/m9-8-morphology.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m9_8_morphology.py
CHECKS = {
 "b": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import no_imports, need, ensure\n\n\ndef blobs():\n    img = (data.coins()[60:140, 20:120] > 120).astype(np.uint8)\n    return img\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/cv2 kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"cv2\", \"scipy\", \"skimage\"}))\n    A = blobs()\n    sym = np.ones((3, 3), np.uint8)\n    asym = np.array([[0, 1, 0], [1, 1, 1], [0, 0, 1]], np.uint8)\n\n    def ero():\n        for B in (sym, asym, np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], np.uint8)):\n            got = np.array(need(ns, \"erode\")(A.tolist(), B.tolist()), np.uint8)\n            want = cv2.erode(A, B, borderType=cv2.BORDER_CONSTANT, borderValue=0)\n            ensure(np.array_equal(got, want), f\"SE {B.tolist()}: cv2.erode ile farklı\")\n    R.test(\"erode = cv2.erode (dışarısı 0)\", ero)\n\n    def dil():\n        got = np.array(need(ns, \"dilate\")(A.tolist(), asym.tolist()), np.uint8)\n        flipped = cv2.dilate(A, asym[::-1, ::-1].copy(), borderType=cv2.BORDER_CONSTANT, borderValue=0)\n        plain = cv2.dilate(A, asym, borderType=cv2.BORDER_CONSTANT, borderValue=0)\n        ensure(np.array_equal(got, flipped), \"ders kitabı genişlemesi = cv2.dilate(A, B[::-1, ::-1]) olmalı\")\n        ensure(np.array_equal(np.array(ns[\"dilate\"](A.tolist(), sym.tolist()), np.uint8), cv2.dilate(A, sym)), \"simetrik SE\")\n        return f\"asimetrik SE: cv2.dilate(A, B) ile {int((got != plain).sum())} piksel farklı, B 180° döndürülünce 0\"\n    R.test(\"dilate (ders kitabı, yansıtılmış B)\", dil)\n\n    def dual():\n        Ac = 1 - A\n        lhs = 1 - np.array(ns[\"erode\"](A.tolist(), asym.tolist()))\n        rhs = np.array(ns[\"dilate\"](Ac.tolist(), asym[::-1, ::-1].tolist()))\n        inner = (slice(2, -2), slice(2, -2))\n        ensure(np.array_equal(lhs[inner], rhs[inner]), \"(A ⊖ B)ᶜ = Aᶜ ⊕ B̂ (kenardan uzakta)\")\n    R.test(\"ikilik\", dual)\n\n    def hmt():\n        I = np.zeros((12, 12), np.uint8); I[2:5, 2:5] = 1; I[6, 2] = 1; I[8:11, 6:10] = 1; I[3, 9] = 1\n        B = [[0, 0, 0], [0, 1, 0], [0, 0, 0]]\n        got = np.array(need(ns, \"hit_or_miss\")(I.tolist(), B))\n        ensure(set(zip(*np.nonzero(got))) == {(6, 2), (3, 9)}, \"tek piksellik nesneler bulunmalı\")\n        C = [[-1, 0, -1], [1, 1, 0], [-1, 1, -1]]\n        got2 = np.array(ns[\"hit_or_miss\"](I.tolist(), C))\n        k = np.array(C); k = np.where(k == -1, 0, np.where(k == 0, -1, 1)).astype(np.int8)\n        want = cv2.morphologyEx(I, cv2.MORPH_HITMISS, k)\n        ensure(np.array_equal(got2[1:-1, 1:-1], want[1:-1, 1:-1]), \"cv2 MORPH_HITMISS ile farklı (iç bölge)\")\n        return f\"sağ üst köşe dedektörü: {list(zip(*np.nonzero(got2)))}\"\n    R.test(\"hit_or_miss (umursamaz elemanlarla)\", hmt)\n\n    def bnd():\n        got = np.array(need(ns, \"boundary\")(A.tolist()), np.uint8)\n        ensure(np.array_equal(got, A - cv2.erode(A, sym, borderType=cv2.BORDER_CONSTANT, borderValue=0)), \"sınır\")\n    R.test(\"boundary\", bnd)\n",
 "c": "import numpy as np\nimport cv2\nfrom scipy import ndimage\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    img = data.camera()\n    ses = {\"kare 5\": np.ones((5, 5), bool), \"elips 7\": cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)).astype(bool), \"artı 3\": np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], bool)}\n    ops = {\"opening\": cv2.MORPH_OPEN, \"closing\": cv2.MORPH_CLOSE, \"gradient\": cv2.MORPH_GRADIENT, \"tophat\": cv2.MORPH_TOPHAT, \"blackhat\": cv2.MORPH_BLACKHAT}\n    for name, se in ses.items():\n        def t(se=se):\n            k = se.astype(np.uint8)\n            ensure(np.array_equal(need(ns, \"gray_erode\")(img, se), cv2.erode(img, k)), \"erozyon\")\n            ensure(np.array_equal(need(ns, \"gray_dilate\")(img, se), cv2.dilate(img, k)), \"genişleme\")\n            for fn, op in ops.items():\n                ensure(np.array_equal(need(ns, fn)(img, se), cv2.morphologyEx(img, op, k)), fn)\n        R.test(f\"7 işlem = cv2 ({name})\", t)\n\n    def asym():\n        se = np.array([[0, 1, 0], [1, 1, 1], [0, 0, 1]], bool)\n        k = se.astype(np.uint8)\n        mine = need(ns, \"opening\")(img, se)\n        cvo = cv2.morphologyEx(img, cv2.MORPH_OPEN, k)\n        ensure(np.array_equal(ns[\"gray_dilate\"](img, se), cv2.dilate(img, k[::-1, ::-1].copy())), \"asimetrik genişleme = cv2.dilate(yansıtılmış)\")\n        ensure(np.all(mine <= img), \"açma görüntüden büyük olamaz (anti-ekstansif)\")\n        ensure(np.array_equal(ns[\"opening\"](mine, se), mine), \"açma idempotent olmalı\")\n        return f\"asimetrik SE: cv2 MORPH_OPEN ile {int((mine != cvo).sum())} piksel farklı; ders kitabı açması anti-ekstansif ve idempotent\"\n    R.test(\"asimetrik SE ve açmanın özellikleri\", asym)\n\n    def nf():\n        bN = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]])\n        got = need(ns, \"nonflat_erode\")(img, bN)\n        want = ndimage.grey_erosion(img.astype(np.float64), structure=bN.astype(np.float64), mode=\"constant\", cval=np.inf)\n        close(got, want, what=\"düz olmayan erozyon\")\n        close(ns[\"nonflat_erode\"](img, np.zeros((3, 3))), cv2.erode(img, np.ones((3, 3), np.uint8)).astype(np.float64), what=\"bN = 0 → düz erozyon\")\n    R.test(\"nonflat_erode\", nf)\n",
 "d": "import numpy as np\nimport cv2\nimport torch\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    img = data.camera().astype(np.float32) / 255\n\n    def flat():\n        asym = np.array([[0, 1, 0], [1, 1, 1], [0, 0, 1]], np.uint8)\n        w = torch.where(torch.tensor(asym) > 0, 0.0, -1e4)\n        got = need(ns, \"dilate_t\")(torch.tensor(img)[None, None], w)[0, 0].numpy()\n        close(got, cv2.dilate(img, asym[::-1, ::-1].copy()), atol=1e-6, what=\"düz asimetrik SE\")\n        bN = torch.tensor([[0.0, 0.1, 0.0], [0.1, 0.2, 0.1], [0.0, 0.1, 0.0]])\n        from scipy import ndimage\n        want = ndimage.grey_dilation(img.astype(np.float64), structure=bN.numpy().astype(np.float64), mode=\"constant\", cval=-1e4)\n        close(ns[\"dilate_t\"](torch.tensor(img)[None, None], bN)[0, 0].numpy(), want, atol=1e-5, what=\"düz olmayan SE\")\n    R.test(\"dilate_t\", flat)\n\n    def learn():\n        cross = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], np.uint8)\n        w = need(ns, \"learn_se\")(img, cross)\n        on, off = w[torch.tensor(cross) > 0], w[torch.tensor(cross) == 0]\n        err = float((ns[\"dilate_t\"](torch.tensor(img)[None, None], w)[0, 0] - torch.tensor(cv2.dilate(img, cross))).abs().max())\n        ensure(on.abs().max() < 0.05, f\"artının elemanları 0 olmalı: {on.tolist()}\")\n        ensure(off.max() < -0.3, f\"köşeler belirgin negatif olmalı: {off.tolist()}\")\n        ensure(err < 0.05, f\"en büyük hata {err:.3f}\")\n        R.table(\"Öğrenilen yapı elemanı (hedef: 3 × 3 artı)\", [\"\", \"sol\", \"orta\", \"sağ\"], [[n] + [f\"{v:+.3f}\" for v in row] for n, row in zip([\"üst\", \"orta\", \"alt\"], w.tolist())],\n                note=f\"En büyük hata {err:.4f}. Köşeler −∞'a gitmez: doğal görüntüde komşu farkları nadiren 0.5'i aşar, köşe 'yeterince' negatif olunca hiç kazanmaz ve gradyan almaz.\")\n    R.test(\"yapı elemanını öğren\", learn)\n\n    def th():\n        u8 = data.coins()\n        ramp = np.linspace(0, 60, u8.shape[1], dtype=np.float32)[None, :]\n        f = np.clip(u8 + ramp, 0, 255).astype(np.float32)\n        got = need(ns, \"tophat_t\")(torch.tensor(f)[None, None], 31)[0, 0].numpy()\n        close(got, cv2.morphologyEx(f, cv2.MORPH_TOPHAT, np.ones((31, 31), np.uint8)), atol=1e-4, what=\"top-hat\")\n        l, r = f[:, :60].mean() - f[:, -60:].mean(), got[:, :60].mean() - got[:, -60:].mean()\n        return f\"sol–sağ ortalama farkı: girdi {l:.1f}, top-hat {r:.1f} (aydınlatma eğimi gitti)\"\n    R.test(\"tophat_t = cv2 MORPH_TOPHAT\", th)\n",
 "e": "import numpy as np\nimport cv2\nimport torch\nimport torch.nn.functional as F\nfrom skimage import data\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n\n    def rm():\n        x = torch.randn(3, 4, 37)\n        for k in (1, 3, 7, 15, 41):\n            want = F.max_pool1d(F.pad(x, (k // 2, k // 2), value=-float(\"inf\")), k, 1)\n            close(need(ns, \"running_max\")(x, k), want, what=f\"k = {k}\")\n    R.test(\"running_max = max_pool1d\", rm)\n\n    img = data.camera()\n    t = torch.tensor(img)\n    for kh, kw in [(3, 3), (15, 7), (61, 61), (1, 31)]:\n        def tt(kh=kh, kw=kw):\n            k = np.ones((kh, kw), np.uint8)\n            ensure(np.array_equal(need(ns, \"rect_dilate\")(t, kh, kw).numpy(), cv2.dilate(img, k)), \"genişleme\")\n            ensure(np.array_equal(need(ns, \"rect_erode\")(t, kh, kw).numpy(), cv2.erode(img, k)), \"erozyon\")\n        R.test(f\"= cv2 ({kh} × {kw})\", tt)\n    if \"rect_dilate\" not in ns:\n        return\n    rows = []\n    for k in (3, 15, 61):\n        naive = lambda: F.unfold(F.pad(t[None, None].float(), (k // 2,) * 4, value=-1.0), k).max(1).values\n        rows.append([k, fmt_time(bench(naive, repeat=3)), fmt_time(bench(lambda: ns[\"rect_dilate\"](t, k, k), repeat=3)), fmt_time(bench(lambda: cv2.dilate(img, np.ones((k, k), np.uint8)), repeat=3))])\n    R.table(\"512 × 512 kare genişleme\", [\"k\", \"unfold (k² eleman/piksel)\", \"van Herk (3 karş./piksel)\", \"cv2.dilate\"], rows,\n            note=\"unfold süresi k² ile büyür; van Herk'inki k'dan bağımsız.\")\n",
 "x1": "import numpy as np\nfrom scipy import ndimage\nfrom skimage import data\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"hazır fonksiyon yok\", lambda: ensure(\"binary_fill_holes\" not in src and \"label(\" not in src, \"binary_fill_holes / label kullanma\"))\n    A = data.coins()[30:180, 20:200] > 110\n\n    def fh():\n        got = need(ns, \"fill_holes\")(A)\n        ensure(np.array_equal(got, ndimage.binary_fill_holes(A)), \"scipy binary_fill_holes ile farklı\")\n        return f\"{int(got.sum() - A.sum())} delik pikseli dolduruldu\"\n    R.test(\"fill_holes\", fh)\n\n    def cc():\n        B = ndimage.binary_opening(A, np.ones((3, 3)))\n        lab = need(ns, \"components\")(B)\n        ref, n = ndimage.label(B, np.ones((3, 3)))\n        ensure(lab.max() == n, f\"{lab.max()} bileşen, {n} bekleniyordu\")\n        firsts = [np.argmax((ref == i).ravel()) for i in range(1, n + 1)]\n        order = np.argsort(firsts)\n        for new, old in enumerate(order, 1):\n            ensure(np.array_equal(lab == new, ref == old + 1), f\"bileşen {new}\")\n        return f\"{n} bileşen\"\n    R.test(\"components (8-bağlantı, satır sırası)\", cc)\n",
 "x2": "import numpy as np\nfrom scipy import ndimage\nfrom skimage import data\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    rows = []\n    for name, A in [(\"dikdörtgen\", np.pad(np.ones((12, 24), bool), 4)),\n                    (\"madeni paralar\", ndimage.binary_opening(data.coins()[40:160, 40:200] > 110, np.ones((3, 3))))]:\n        def t(A=A, name=name):\n            for Bn, B in [(\"kare\", np.ones((3, 3), bool)), (\"artı\", ndimage.generate_binary_structure(2, 1))]:\n                S, parts = need(ns, \"skeleton\")(A, B)\n                ensure(np.all(S <= A), \"iskelet A'nın içinde olmalı\")\n                E = A.copy(); K = 0\n                while ndimage.binary_erosion(E, B).any():\n                    E = ndimage.binary_erosion(E, B); K += 1\n                ensure(len(parts) == K + 1, f\"{len(parts)} parça, {K + 1} bekleniyordu\")\n                ensure(np.array_equal(need(ns, \"reconstruct\")(parts, B), A), \"geri çatma A'yı vermedi\")\n                rows.append([name, Bn, int(A.sum()), int(S.sum()), K])\n        R.test(f\"iskelet ve geri çatma ({name})\", t)\n    if rows:\n        R.table(\"Lantuéjoul iskeleti\", [\"nesne\", \"SE\", \"|A|\", \"|S|\", \"K\"], rows, note=\"İskelet bağlantılı olmak zorunda değildir; ama S_k'ler ve k'lar A'yı tam olarak geri verir.\")\n",
 "x3": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef disks_image(radii, seed=0):\n    rng = np.random.default_rng(seed)\n    img = np.zeros((256, 256), np.uint8)\n    for r in radii:\n        for _ in range(6):\n            cv2.circle(img, (int(rng.integers(r, 256 - r)), int(rng.integers(r, 256 - r))), r, 200, -1)\n    return img\n\n\ndef check(ns, R):\n    radii = list(range(1, 24))\n\n    def gran():\n        img = disks_image([12])\n        g = need(ns, \"granulometry\")(img, radii)\n        ensure(len(g) == len(radii) and g[0] > g[-1], \"açma büyüdükçe toplam azalmalı\")\n        ps = need(ns, \"pattern_spectrum\")(img, radii)\n        close(ps.sum(), 1.0, what=\"toplam 1\")\n        peak = radii[int(np.argmax(ps)) + 1]\n        ensure(abs(peak - 13) <= 1, f\"tepe yarıçapı {peak}, ≈ 12–13 bekleniyordu\")\n        two = ns[\"pattern_spectrum\"](disks_image([5, 15], 1), radii)\n        top2 = sorted(radii[i + 1] for i in np.argsort(two)[-2:])\n        return f\"tek boyut: tepe r = {peak}; iki boyut: tepeler r = {top2}\"\n    R.test(\"granülometri ve desen spektrumu\", gran)\n\n    def sm():\n        clean = data.camera()\n        rng = np.random.default_rng(0)\n        noisy = clean.copy(); u = rng.random(clean.shape); noisy[u < 0.03] = 0; noisy[u > 0.97] = 255\n        out = need(ns, \"oc_smooth\")(noisy, 1)\n        p0, p1 = cv2.PSNR(clean, noisy), cv2.PSNR(clean, out)\n        ensure(p1 > p0 + 5, f\"PSNR {p0:.1f} → {p1:.1f}\")\n        return f\"tuz-biber %6: PSNR {p0:.1f} → {p1:.1f} dB (median 3×3: {cv2.PSNR(clean, cv2.medianBlur(noisy, 3)):.1f})\"\n    R.test(\"açma–kapama yumuşatma\", sm)\n"
}

In [ ]:
import importlib, labcheck, checks_m9_8_morphology as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Kümeler ve yapı elemanları (SE):** ikili görüntüde nesne $A$ = ön plan pikselleri. SE küçük bir kümedir, bir **orijini** vardır (genelde merkez). Yansıma $\hat B=\{-b\mid b\in B\}$ (180° döndürme), öteleme $(B)_z=\{b+z\}$.

**Erozyon:** $A\ominus B=\{z\mid(B)_z\subseteq A\}$ — SE'nin tamamen sığdığı yerler; nesneler küçülür, $B$'den ince bağlar kopar.

**Genişleme:** $A\oplus B=\{z\mid(\hat B)_z\cap A\ne\varnothing\}=\{a+b\}$ (Minkowski toplamı); nesneler büyür, boşluklar dolar.

**İkilik:** $(A\ominus B)^c=A^c\oplus\hat B$, $(A\oplus B)^c=A^c\ominus\hat B$.

**Açma** $A\circ B=(A\ominus B)\oplus B=\bigcup\{(B)_z\mid(B)_z\subseteq A\}$: konturu yumuşatır, ince çıkıntıları ve köprüleri siler. **Kapama** $A\bullet B=(A\oplus B)\ominus B$: dar boşlukları ve küçük delikleri kapatır. Özellikler: $A\circ B\subseteq A\subseteq A\bullet B$; ikisi de **idempotent** ($(A\circ B)\circ B=A\circ B$).

**hit-or-miss:** $I\circledast B_{1,2}=\{z\mid(B_1)_z\subseteq A\ \text{ve}\ (B_2)_z\subseteq A^c\}$; tek SE'de ön plan (1), arka plan (0) ve **umursamaz** elemanlarla da yazılır. Algoritmalar: sınır $A-(A\ominus B)$, delik doldurma, bağlı bileşenler, dışbükey zarf, inceltme, iskelet, budama.

**Gri seviye**, düz SE: erozyon $=\min_{(s,t)\in b}f(x+s,y+t)$, genişleme $=\max_{(s,t)\in b}f(x-s,y-t)$. Düz olmayan: $[f\ominus b_N]=\min(f(x+s,y+t)-b_N(s,t))$, $[f\oplus b_N]=\max(f(x-s,y-t)+b_N(s,t))$. **Gradyan** $(f\oplus b)-(f\ominus b)$; **top-hat** $f-(f\circ b)$; **bottom-hat** $(f\bullet b)-f$. Geometrik yorum: açma, SE'yi yüzeyin **altından** yukarı itip ulaştığı en yüksek noktaları alır.

## 1 · Elle

Kare: erozyon $5-2=3$ → 9 piksel; genişleme $5+2=7$ → 49. Artı ile genişlemede köşeler dolmaz: $49-4=45$.

Açma örneğinde 2'lik dizi SE'ye sığmaz, silinir; 3'lük kalır.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**9 × 9 görüntüde 5 × 5 kare $A$ (dışarısı arka plan)**

- `e33`: $A\ominus B$, $B$ = 3 × 3 kare: ön plan piksel sayısı
- `d33`: $A\oplus B$, 3 × 3 kare
- `ecr`: $A\ominus B$, $B$ = 3 × 3 artı (+)
- `dcr`: $A\oplus B$, artı
- `bnd`: Sınır $A-(A\ominus B)$, 3 × 3 kare

**1-B, düz yapı elemanı uzunluk 3 (merkezli): $f=[3, 7, 2, 9, 4]$**

- `ge`: Erozyon, indis 2
- `gd`: Genişleme, indis 1
- `gg`: Morfolojik gradyan, indis 2

**Diğer**

- `nf`: Düz olmayan erozyon: $f=[5,8,6]$, $b_N=[1,2,1]$, merkezde $\min(f-b_N)$
- `op`: İkili $[0,1,1,0,1,1,1,0,1]$'in uzunluk 3 ile açılması: kaç 1 kalır?

In [ ]:
cevaplar = dict(
    e33=None,
    d33=None,
    ecr=None,
    dcr=None,
    bnd=None,
    ge=None,
    gd=None,
    gg=None,
    nf=None,
    op=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"e33":[9,0],"d33":[49,0],"ecr":[9,0],"dcr":[45,0],"bnd":[16,0],"ge":[2,0],"gd":[7,0],"gg":[7,0],"nf":[4,0],"op":[3,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, ikili (0/1) liste listeleri. SE: tek boyutlu kare liste, orijin merkezde; 1 = eleman. Görüntü dışı **arka plan** (0).

- `erode(A, B)` — $A\ominus B$.
- `dilate(A, B)` — **ders kitabı** tanımı $A\oplus B=\{z\mid(\hat B)_z\cap A\ne\varnothing\}$, yani $z$'de sonuç 1 ⇔ bir $b\in B$ için $A(z-b)=1$.
- `hit_or_miss(I, B)` — `B` elemanları 1 (ön plan olmalı), 0 (arka plan olmalı), −1 (umursamaz).
- `boundary(A)` — $A-(A\ominus B)$, $B$ = 3 × 3 kare.

Kontrol, `cv2.dilate`'in çekirdeği **yansıtmadan** max aldığını gösterir: asimetrik SE'de seninki yalnız ters çevrilmiş çekirdekle eşleşir.

In [ ]:
%%writefile stage_b.py
def erode(A, B):
    ...


def dilate(A, B):
    ...


def hit_or_miss(I, B):
    ...


def boundary(A):
    ...


A = [[0] * 9 for _ in range(9)]
for y in range(2, 7):
    for x in range(2, 7):
        A[y][x] = 1
print(sum(map(sum, erode(A, [[1] * 3] * 3))))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok**, `uint8` gri seviye; `np.lib.stride_tricks.sliding_window_view`. SE `bool` maske (tek boyutlu), orijin merkezde. Kenar kuralı OpenCV'nin varsayılanı: erozyonda dışarısı **255**, genişlemede **0** (yani dışarısı hiç seçilmez).

- `gray_erode(f, se)`, `gray_dilate(f, se)` (ders kitabı: yansıtılmış SE ile max).
- `opening(f, se)`, `closing(f, se)`, `gradient(f, se)`, `tophat(f, se)`, `blackhat(f, se)`.
- `nonflat_erode(f, bN)` — `bN` tamsayı dizisi; `float64` döndür, dışarısı $+\infty$.

Simetrik SE'lerde `cv2.morphologyEx` ile birebir; asimetrikte `cv2.morphologyEx(MORPH_OPEN)` ders kitabından ayrılır (neden?).

In [ ]:
%%writefile stage_c.py
import numpy as np


def gray_erode(f, se):
    ...


def gray_dilate(f, se):
    ...


def opening(f, se):
    ...


def closing(f, se):
    ...


def gradient(f, se):
    ...


def tophat(f, se):
    ...


def blackhat(f, se):
    ...


def nonflat_erode(f, bN):
    ...


from skimage import data
print(gradient(data.camera(), np.ones((3, 3), bool)).mean())

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Morfoloji = toplama yerine max/min alan "konvolüsyon".

- `dilate_t(f, w)` — `f (B, 1, H, W)` float, `w (k, k)`: düz olmayan **ders kitabı** genişlemesi $\max_{b}[f(z-b)+w(b)]$. `F.unfold` ile; dışarısı $-10^4$. Düz SE: `w`'de eleman 0, eleman olmayan $-\infty$ (pratikte çok negatif).
- `learn_se(img, target_se, steps=1500, lr=0.02, patch=128, seed=0)` — sıfırdan başlayan 3 × 3 `w`'yu, rastgele yamalarda `dilate_t(f, w)` ile `cv2.dilate(img, target_se)` arasındaki MSE ile Adam'la öğren; `w`'yu döndür.
- `tophat_t(f, k)` — kare $k\times k$ düz SE ile top-hat; erozyon `-max_pool2d(-f)`, genişleme `max_pool2d` (dolgu $-\infty$ olduğundan OpenCV kenar kuralıyla aynı).

Gradyan yalnız `max`'ı kazanan elemana akar: öğrenme eleman olmayan köşeleri "asla kazanamayacakları" kadar aşağı iter.

In [ ]:
%%writefile stage_d.py
import numpy as np
import torch
import torch.nn.functional as F


def dilate_t(f, w):
    ...


def learn_se(img, target_se, steps=1500, lr=0.02, patch=128, seed=0):
    ...


def tophat_t(f, k):
    ...


f = torch.rand(1, 1, 8, 8)
print(dilate_t(f, torch.zeros(3, 3)).shape)

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

**van Herk / Gil-Werman:** uzunluğu $k$ olan pencerede max, eleman başına **3 karşılaştırma** ile, $k$'dan bağımsız. Diziyi $k$'lık bloklara böl; her blokta soldan kümülatif max $g$ ve sağdan kümülatif max $h$. $[i,i+k-1]$ penceresi en fazla iki bloğa yayılır: $\max(h_i, g_{i+k-1})$.

- `running_max(x, k)` — `x (..., L)` float, merkezli tek $k$, dışarısı $-\infty$; son eksen boyunca. `torch.cummax`, `flip`, `reshape` ile, döngüsüz.
- `rect_dilate(img, kh, kw)`, `rect_erode(img, kh, kw)` — `uint8` tensör, `cv2.dilate/erode(img, np.ones((kh, kw)))` ile birebir (ayrılabilirlik: önce satırlar, sonra sütunlar). Erozyon için $\min(x)=-\max(-x)$.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F


def running_max(x, k):
    ...


def rect_dilate(img, kh, kw):
    ...


def rect_erode(img, kh, kw):
    ...


print(running_max(torch.tensor([1.0, 5, 2, 0, 3, 9, 1]), 3))

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Morfolojik algoritmalar: delik doldurma ve bağlı bileşenler

Ders kitabı algoritmaları (NumPy; `scipy.ndimage.binary_dilation` serbest, `binary_fill_holes` ve `label` yasak):

- `fill_holes(A)` — $X_0$: görüntü kenarındaki arka plan pikselleri; $X_k=(X_{k-1}\oplus B)\cap A^c$ (3 × 3 artı) değişmeyene dek; sonuç $X_k^c$. (Kenardan ulaşılamayan arka plan = delik.)
- `components(A)` — `int32` etiket haritası: etiketlenmemiş ilk pikselden $X_k=(X_{k-1}\oplus B)\cap A$ (3 × 3 kare, 8-bağlantı) ile bileşeni çıkar, satır sırasına göre 1, 2, … numarala.

In [ ]:
%%writefile extra_x1.py
import numpy as np
from scipy import ndimage


def fill_holes(A):
    ...


def components(A):
    ...


A = np.zeros((7, 7), bool); A[1:6, 1:6] = True; A[3, 3] = False
print(fill_holes(A).sum())

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · İskelet (Lantuéjoul) ve geri çatma

**Lantuéjoul iskeleti:** $S(A)=\bigcup_{k=0}^{K}S_k$, $S_k=(A\ominus kB)-(A\ominus kB)\circ B$ ($kB$: $k$ kez art arda erozyon), $K$ = $A$ boşalmadan önceki son $k$. **Geri çatma:** $A=\bigcup_k S_k\oplus kB$.

- `skeleton(A, B)` → `(S, parts)`: birleşim ve $S_k$ listesi.
- `reconstruct(parts, B)` — $A$'yı geri ver.

`scipy.ndimage` ikili işlemleri serbest (kenar kuralı: dışarısı arka plan).

In [ ]:
%%writefile extra_x2.py
import numpy as np
from scipy import ndimage


def skeleton(A, B):
    ...


def reconstruct(parts, B):
    ...


A = np.zeros((20, 30), bool); A[4:16, 3:27] = True
S, parts = skeleton(A, np.ones((3, 3), bool))
print(S.sum(), len(parts))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Granülometri ve açma–kapama ile yumuşatma

- `granulometry(img, radii)` — her $r$ için disk (`cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2r+1, 2r+1))`) ile açılmış görüntünün toplam yoğunluğu.
- `pattern_spectrum(img, radii)` — ardışık farkların negatifi (toplamı 1'e normalize). Tepe noktası baskın parça yarıçapını verir. (Sayısal diskler iç içe olmadığından küçük negatif değerler çıkabilir: kuramda açmalar $r$ ile monoton azalır, ızgarada her zaman değil.)
- `oc_smooth(img, r)` — önce açma sonra kapama (disk); tuz-biber ve küçük parlak/koyu gürültüyü temizler.

In [ ]:
%%writefile extra_x3.py
import numpy as np
import cv2


def granulometry(img, radii):
    ...


def pattern_spectrum(img, radii):
    ...


def oc_smooth(img, r):
    ...


from skimage import data
print(pattern_spectrum(data.coins(), range(1, 30, 2)).argmax())

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Morfolojik ağlar: genişleme–erozyon katmanları

[Mondal, Santra, Chanda — Dense Morphological Network: An Universal Function Approximator (2019)](https://arxiv.org/abs/1901.00109)

Toplama–max (genişleme) ve toplama–min (erozyon) nöronlarından oluşan yoğun bir morfolojik katman (DenMo-Net) uygula; tek gizli katmanlı sürümün bir fonksiyon yaklaşıklayıcı olduğunu deneyle göster ve MNIST / Fashion-MNIST'te aynı parametre sayısındaki ReLU MLP ile karşılaştır.


**Adımlar**

1. Genişleme nöronu $\max_i(x_i+w_i)$ ve erozyon nöronu $\min_i(x_i-w_i)$; bu modüldeki öğrenilebilir yapı elemanını genelleştir.
2. Gradyanın yalnız argmax'a aktığını gözlemle; seyrek gradyan için öğrenme oranı ve ilklendirme deneyleri.
3. 2-B oyuncak sınıflandırma (iç içe halkalar) karar sınırları; gizli nöron sayısının etkisi.
4. Konvolüsyonel sürüm: 3 × 3 morfolojik filtrelerle küçük bir ağ.

**Kabul ölçütleri**

- Halka verisinde %98 üstü doğruluk ve karar sınırı görseli.
- Fashion-MNIST'te morfolojik ağ ve MLP doğruluk/parametre tablosu.
- Öğrenilen yapı elemanlarının görselleştirilmesi.

## Kaynaklar

- [Gonzalez & Woods — Digital Image Processing, 4. baskı, Bölüm 9](https://www.imageprocessingplace.com/) — ©
- [OpenCV — erode, dilate, morphologyEx](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html) — Apache-2.0
- [van Herk — A fast algorithm for local minimum and maximum filters on rectangular and octagonal kernels (PRL 1992)](https://doi.org/10.1016/0167-8655(92)90069-C) — ©
- [scikit-image — morphology](https://scikit-image.org/docs/stable/api/skimage.morphology.html) — BSD-3